# 04 — Feature Selection

**Primary:** Zhilin Zhang  
**Support:** Tianyi Qin

This notebook applies:
- **embedded method:** Decision Tree feature importance;
- **filter method:** Mutual Information with the high-price target.

Both methods use the same full feature set and the same training rows created by preprocessing.

## 1. Imports and data

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"
MODEL_SUMMARY_PATH = REPO_ROOT / "output" / "tables" / "model_summary.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_preprocessing.ipynb first.")
if not MODEL_SUMMARY_PATH.exists():
    raise FileNotFoundError("Run 03_modelling.ipynb first.")

df = pd.read_csv(DATA_PATH)
model_summary = pd.read_csv(MODEL_SUMMARY_PATH)

train_df = df.loc[df["split"].eq("train")].copy()
test_df = df.loc[df["split"].eq("test")].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

## 2. Full feature set

In [ ]:
SIZE_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
]
LOCATION_FEATURES = [
    "distance_cbd_km",
]
AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]
FEATURES = SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES
TARGET = "high_price"

missing = [c for c in FEATURES + [TARGET] if c not in train_df.columns]
if missing:
    raise KeyError(f"Missing feature-selection columns: {missing}")

## 3. Embedded method — tuned Decision Tree importance

The tuned full-feature Decision Tree settings are loaded from the modelling output so feature selection is traceable to the same model comparison.

In [ ]:
tree_row = model_summary[
    (model_summary["model"] == "DecisionTree")
    & (model_summary["feature_set"] == "size_location_amenities")
].iloc[0]

best_params = json.loads(tree_row["best_params"])

tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])
tree_pipeline.set_params(**best_params)
tree_pipeline.fit(train_df[FEATURES], train_df[TARGET])

tree_model = tree_pipeline.named_steps["model"]

embedded_scores = pd.Series(
    tree_model.feature_importances_,
    index=FEATURES,
    name="embedded_score",
).sort_values(ascending=False)

embedded_top3 = embedded_scores.head(3)
display(embedded_top3.to_frame())

## 4. Filter method — Mutual Information

Missing numeric values are median-imputed using training data only. Binary amenity indicators are explicitly marked as discrete; the remaining count/continuous variables are treated as continuous by `mutual_info_classif`.

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(train_df[FEATURES])
y_train = train_df[TARGET].to_numpy()

discrete_mask = np.array([
    feature.startswith("has_")
    for feature in FEATURES
], dtype=bool)

mi_scores = mutual_info_classif(
    X_train_imputed,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE,
)

filter_scores = pd.Series(
    mi_scores,
    index=FEATURES,
    name="filter_mi_score",
).sort_values(ascending=False)

filter_top3 = filter_scores.head(3)
display(filter_top3.to_frame())

## 5. Compare the two top-3 lists

In [ ]:
comparison = pd.DataFrame({
    "embedded_feature": embedded_top3.index.to_list(),
    "embedded_score": embedded_top3.values,
    "filter_feature": filter_top3.index.to_list(),
    "filter_mi_score": filter_top3.values,
})

display(comparison)

all_rankings = pd.DataFrame({
    "feature": FEATURES,
    "embedded_score": [embedded_scores[f] for f in FEATURES],
    "embedded_rank": [int(embedded_scores.index.get_loc(f) + 1) for f in FEATURES],
    "filter_mi_score": [filter_scores[f] for f in FEATURES],
    "filter_rank": [int(filter_scores.index.get_loc(f) + 1) for f in FEATURES],
})
all_rankings["rank_difference"] = (
    all_rankings["embedded_rank"] - all_rankings["filter_rank"]
).abs()

display(
    all_rankings.sort_values(
        ["rank_difference", "embedded_rank"],
        ascending=[False, True],
    )
)

## 6. Identify a hard-case listing

The rubric allows a listing that is a genuine outlier on a top-ranked feature. The code selects the first top-ranked feature that is numeric/non-binary and identifies the listing furthest beyond the 1.5×IQR bounds. It then exports the listing ID and its actual relevant attributes.

In [ ]:
ranked_candidates = []
for feature in list(embedded_top3.index) + list(filter_top3.index):
    if feature not in ranked_candidates:
        ranked_candidates.append(feature)

hard_feature = None
hard_candidates = None
hard_bounds = None

for feature in ranked_candidates:
    values = pd.to_numeric(df[feature], errors="coerce")
    if values.nunique(dropna=True) <= 2:
        continue

    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    distance_outside = pd.Series(0.0, index=df.index)
    distance_outside.loc[values < low] = low - values.loc[values < low]
    distance_outside.loc[values > high] = values.loc[values > high] - high

    mask = distance_outside > 0
    if mask.any():
        hard_feature = feature
        hard_bounds = {
            "q1": float(q1),
            "q3": float(q3),
            "iqr": float(iqr),
            "lower_bound": float(low),
            "upper_bound": float(high),
        }
        hard_candidates = df.loc[mask].copy()
        hard_candidates["outlier_distance"] = distance_outside.loc[mask]
        hard_candidates = hard_candidates.sort_values(
            "outlier_distance",
            ascending=False,
        )
        break

if hard_feature is None:
    raise RuntimeError(
        "No IQR outlier found among non-binary top-ranked features; "
        "select a model-disagreement hard case instead."
    )

hard_row = hard_candidates.iloc[0]

hard_case_columns = [
    "id",
    "split",
    "high_price",
    "price_clean",
] + FEATURES

hard_case = hard_row[hard_case_columns].to_frame().T
hard_case.insert(0, "hard_case_feature", hard_feature)
hard_case["typical_q1"] = hard_bounds["q1"]
hard_case["typical_q3"] = hard_bounds["q3"]
hard_case["iqr_lower_bound"] = hard_bounds["lower_bound"]
hard_case["iqr_upper_bound"] = hard_bounds["upper_bound"]

print("Hard-case top-ranked feature:", hard_feature)
print("Typical bounds:", hard_bounds)
display(hard_case)

## 7. Save feature-selection outputs

In [ ]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

comparison.to_csv(TABLE_OUT / "feature_selection_top3.csv", index=False)
all_rankings.to_csv(TABLE_OUT / "feature_selection_all_rankings.csv", index=False)
hard_case.to_csv(TABLE_OUT / "feature_selection_hard_case.csv", index=False)

plot_rankings = all_rankings.sort_values("embedded_score", ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(plot_rankings["feature"], plot_rankings["embedded_score"])
ax.set_xlabel("Decision Tree embedded importance")
ax.set_title("Embedded feature importance")
fig.tight_layout()
fig.savefig(FIG_OUT / "feature_selection_embedded.png", dpi=200)
plt.close(fig)

plot_filter = all_rankings.sort_values("filter_mi_score", ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(plot_filter["feature"], plot_filter["filter_mi_score"])
ax.set_xlabel("Mutual Information score")
ax.set_title("Filter feature importance")
fig.tight_layout()
fig.savefig(FIG_OUT / "feature_selection_filter_mi.png", dpi=200)
plt.close(fig)

print("Saved feature-selection outputs to:", TABLE_OUT)

## 8. Evidence checklist for group-written interpretation

Use the generated outputs to explain:
- where the embedded and filter top-3 lists agree/disagree;
- why the two methods can rank a feature differently in this particular feature set;
- the hard-case listing ID and its actual attributes;
- why that row is difficult/unusual relative to the typical range;
- limitations such as correlated size variables splitting tree importance and MI being univariate.